[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/onnx_torchscript.ipynb)

# 05 . TorchScript, torch.export and ONNX

## Concept — Model Export (TorchScript, torch.export, ONNX) (revision notes)

**What it is**
- Exporting turns a Python `nn.Module` into a portable, Python-independent graph representation that can be saved, optimized and run elsewhere.
- TorchScript (`torch.jit.trace` / `torch.jit.script`) is the older PyTorch-native format; `torch.export` is the newer graph capture that produces an `ExportedProgram`; ONNX is a cross-framework format run by ONNX Runtime, TensorRT, etc.

**Why it matters**
- Deploying to C++ servers, mobile or other runtimes without shipping Python and the model's source code.
- Graph-level optimizations (operator fusion, constant folding) and hardware-specific runtimes can beat eager execution.
- Export is where eager-mode assumptions break: data-dependent control flow, Python side effects and dynamic shapes all need care.

**When to use**
- `torch.jit.trace`: models with no data-dependent control flow (typical CNN/MLP).
- `torch.jit.script`: models with `if`/loops depending on tensor values (but code must be TorchScript-compatible).
- `torch.export`: the forward-looking path for `torch.compile`, AOTInductor and ExecuTorch.
- ONNX: when the target runtime is not PyTorch.

**Math & intuition**
- Tracing runs the model once on an example input and RECORDS the tensor ops executed: Python `if` statements are baked in as whichever branch happened to run.
- Scripting compiles the Python source (a restricted subset) into IR, so control flow is preserved.
- NOTE: TorchScript is deprecated in recent PyTorch releases (maintenance mode); it is still widely deployed, so you must understand it, but new projects should prefer `torch.export`/ONNX.
- Always verify an exported model numerically against eager on several inputs (including different batch sizes) with `torch.testing.assert_close`.

### 1. A model that has data-dependent control flow

**What this cell does (plain language):** we define a small module with an `if` that depends on the input's values, plus a normal sub-network. This is the standard example that distinguishes tracing from scripting. We also run it in eager mode on two inputs that take different branches.

In [1]:
import os, io, tempfile, warnings, time, logging, contextlib
import torch, torch.nn as nn
# TorchScript is in maintenance mode and recent torch versions emit FutureWarnings for it;
# we silence them to keep output readable (read the note in the concept section!).
warnings.filterwarnings("ignore", message=".*is deprecated.*", category=FutureWarning)
logging.getLogger("torch.fx.experimental.symbolic_shapes").setLevel(logging.ERROR)
torch.manual_seed(0)

class Gate(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))
    def forward(self, x):
        h = self.body(x)
        if x.mean() > 0:          # DATA-DEPENDENT branch (depends on the input values)
            return h * 2.0
        else:
            return h - 1.0

m = Gate().eval()
x_pos = torch.ones(2, 8); x_neg = -torch.ones(2, 8) * 5
print("input means:", x_pos.mean().item(), x_neg.mean().item(), "-> different branches")
print("eager outputs, first 2 values:", m(x_pos)[0, :2].tolist(), m(x_neg)[0, :2].tolist())
print("(x_pos branch = h*2, x_neg branch = h-1, so the two paths compute different functions)")

input means: 1.0 -5.0 -> different branches
eager outputs, first 2 values: [0.020241186022758484, -0.625251829624176] [-3.586045980453491, -0.024765491485595703]
(x_pos branch = h*2, x_neg branch = h-1, so the two paths compute different functions)


### 2. torch.jit.trace: fast, but it freezes the branch

**What this cell does (plain language):** we trace the model with `x_pos`. PyTorch warns (we capture the warning to show it) that the data-dependent `if` was converted to a constant. Then we feed an input that would take the OTHER branch in eager mode and see that the traced model silently produces the wrong answer.

In [2]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    traced = torch.jit.trace(m, x_pos)
# keep only the TracerWarning about the data-dependent condition (not the deprecation notes)
print("tracer warnings:", [str(i.message)[:110] for i in w if "Converting a tensor to a Python boolean" in str(i.message)][:1])

for name, xi in (("x_pos", x_pos), ("x_neg", x_neg)):
    e, t = m(xi), traced(xi)
    print(f"{name}: traced == eager ? {torch.allclose(e, t)}")

tracer warnings: ["Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow o"]
x_pos: traced == eager ? True
x_neg: traced == eager ? False


### 3. torch.jit.script: preserves control flow

**What this cell does (plain language):** we compile the same module with `torch.jit.script` and repeat the comparison. Now both branches are present in the compiled graph, so the output matches eager on both inputs. We print the generated TorchScript `code` to see the `if` survived.

In [3]:
scripted = torch.jit.script(m)
for name, xi in (("x_pos", x_pos), ("x_neg", x_neg)):
    print(f"{name}: scripted == eager ? {torch.allclose(m(xi), scripted(xi))}")
print(scripted.code)

x_pos: scripted == eager ? True
x_neg: scripted == eager ? True
def forward(self,
    x: Tensor) -> Tensor:
  body = self.body
  h = (body).forward(x, )
  if bool(torch.gt(torch.mean(x), 0)):
    _0 = torch.mul(h, 2.)
  else:
    _0 = torch.sub(h, 1.)
  return _0



### 4. Saving, loading and running without the Python class

**What this cell does (plain language):** we save the scripted module to an in-memory buffer (a file works identically), delete our `Gate` class definition from the namespace, and load it back. The archive contains code AND weights, so no Python class is needed — this is how a C++ app can run the model with `torch::jit::load`.

In [4]:
buf = io.BytesIO()
torch.jit.save(scripted, buf); buf.seek(0)
size = len(buf.getvalue())
del Gate                                   # the class is gone from this namespace
loaded = torch.jit.load(buf)
print("archive size (bytes):", size, "| type:", type(loaded).__name__)
print("loaded == eager (both branches):", all(torch.allclose(m(i), loaded(i)) for i in (x_pos, x_neg)))

archive size (bytes): 9871 | type: RecursiveScriptModule
loaded == eager (both branches): True


### 5. Comparing outputs properly: a reusable checker

**What this cell does (plain language):** exports must be verified, not trusted. We write `check()` that runs several random inputs (including different batch sizes) through the reference and candidate and uses `torch.testing.assert_close`, reporting the largest absolute difference. We use it on a trace-friendly model (no data-dependent branches).

In [5]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(8, 32), nn.Tanh(), nn.Linear(32, 4))
    def forward(self, x): return self.f(x)

net = Net().eval()

def check(ref, cand, shapes=((1, 8), (3, 8), (16, 8)), atol=1e-5, name="candidate"):
    worst = 0.0
    for s in shapes:
        xi = torch.randn(*s)
        with torch.no_grad():
            r, c = ref(xi), cand(xi)
        torch.testing.assert_close(c, r, atol=atol, rtol=1e-4)
        worst = max(worst, (r - c).abs().max().item())
    print(f"{name}: OK on batch sizes {[s[0] for s in shapes]} | max abs diff {worst:.2e}")

tr_net = torch.jit.trace(net, torch.randn(2, 8))
check(net, tr_net, name="trace")
check(net, torch.jit.script(net), name="script")

trace: OK on batch sizes [1, 3, 16] | max abs diff 0.00e+00
script: OK on batch sizes [1, 3, 16] | max abs diff 0.00e+00


### 6. Tracing pitfall: hard-coded shapes and train/eval mode

**What this cell does (plain language):** we show two classic export traps. (1) A model that reads `x.shape[0]` in Python and builds a tensor from it: tracing bakes in the batch size seen at trace time. (2) Tracing in `train()` mode records dropout as an active op, so the 'exported' model is random at inference time.

In [6]:
class Shapey(nn.Module):
    def forward(self, x):
        n = int(x.shape[0])                   # converting to a Python int freezes it during tracing
        return torch.ones(n, 1) * x.sum(0, keepdim=True)   # eager: (batch, 3)
xi = torch.randn(10, 3)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")            # hide the TracerWarning that PyTorch emits for int(shape)
    sh_tr = torch.jit.trace(Shapey(), torch.randn(4, 3))
    print("eager output shape :", tuple(Shapey()(xi).shape))
print("traced output shape:", tuple(sh_tr(xi).shape), "<- batch size 4 was baked in")

drop = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5))
drop.train()
tr_train = torch.jit.trace(drop, torch.randn(2, 4), check_trace=False)   # skip the check: it would complain too
z = torch.randn(2, 4)
print("train-mode trace deterministic?", torch.equal(tr_train(z), tr_train(z)))   # False: dropout baked in
drop.eval(); tr_eval = torch.jit.trace(drop, torch.randn(2, 4))
print("eval-mode  trace deterministic?", torch.equal(tr_eval(z), tr_eval(z)))

eager output shape : (10, 3)
traced output shape: (4, 3) <- batch size 4 was baked in
train-mode trace deterministic? False
eval-mode  trace deterministic? True


### 7. torch.export: the modern graph capture

**What this cell does (plain language):** `torch.export.export` captures a full `ExportedProgram` (an FX graph in ATen ops with explicit input/output signature). Unlike `trace` it errors on constructs it cannot capture rather than silently baking them in. We declare the batch dimension dynamic with `torch.export.Dim`, print the graph, and verify against eager at several batch sizes.

In [7]:
from torch.export import export, Dim
batch = Dim("batch", min=1, max=64)
ex_inp = torch.randn(4, 8)
ep = export(net, (ex_inp,), dynamic_shapes={"x": {0: batch}})
print("type:", type(ep).__name__)
print(ep.graph_module.code)
check(net, ep.module(), shapes=((1, 8), (7, 8), (64, 8)), name="torch.export (dynamic batch)")

type: ExportedProgram



def forward(self, p_f_0_weight, p_f_0_bias, p_f_2_weight, p_f_2_bias, x):
    linear = torch.ops.aten.linear.default(x, p_f_0_weight, p_f_0_bias);  x = p_f_0_weight = p_f_0_bias = None
    tanh = torch.ops.aten.tanh.default(linear);  linear = None
    linear_1 = torch.ops.aten.linear.default(tanh, p_f_2_weight, p_f_2_bias);  tanh = p_f_2_weight = p_f_2_bias = None
    return (linear_1,)
    
torch.export (dynamic batch): OK on batch sizes [1, 7, 64] | max abs diff 0.00e+00


### 8. torch.export on data-dependent control flow

**What this cell does (plain language):** we try to export the `Gate` model again (we rebuild the class first). `torch.export` cannot decide a branch on a tensor value at export time, so it raises a clear error instead of tracing one branch. The fix is `torch.cond`, which expresses the branch as an explicit graph operator.

In [8]:
class Gate(nn.Module):
    def __init__(self):
        super().__init__(); self.body = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))
    def forward(self, x):
        h = self.body(x)
        if x.mean() > 0: return h * 2.0
        return h - 1.0

class GateCond(Gate):
    def forward(self, x):
        h = self.body(x)
        # torch.cond(pred, true_fn, false_fn, operands): both branches live in the exported graph
        return torch.cond(x.mean() > 0, lambda h: h * 2.0, lambda h: h - 1.0, (h,))

g1 = Gate().eval()
sink = io.StringIO()                       # torch logs a long graph dump on failure; hide it
try:
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        export(g1, (x_pos,))
    print("export of the plain `if` unexpectedly succeeded")
except Exception as e:
    print("plain `if` ->", type(e).__name__)

g2 = GateCond().eval(); g2.load_state_dict(g1.state_dict())
ep2 = export(g2, (x_pos,))
print("torch.cond export matches eager on both branches:",
      all(torch.allclose(ep2.module()(i), g1(i)) for i in (x_pos, x_neg)))

plain `if` -> GuardOnDataDependentSymNode


torch.cond export matches eager on both branches: True


### 9. ONNX export (guarded: needs the `onnx` package)

**What this cell does (plain language):** we export the MLP to ONNX with a dynamic batch axis, then validate the file with `onnx.checker`. If `onnx` (or `onnxscript`, which the new exporter requires) is not installed, we print the install command and move on, so the notebook still runs top to bottom.

In [9]:
onnx_path = os.path.join(tempfile.mkdtemp(), "net.onnx")
onnx_ok = False
sink = io.StringIO()
try:
    import onnx
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):   # hide exporter progress chatter
        torch.onnx.export(net, (torch.randn(2, 8),), onnx_path,
                          input_names=["x"], output_names=["y"],
                          dynamic_shapes={"x": {0: "batch"}})   # batch size stays flexible
    # (older PyTorch / dynamo=False exporter: use dynamic_axes={"x": {0: "batch"}, "y": {0: "batch"}})
    model_proto = onnx.load(onnx_path)
    onnx.checker.check_model(model_proto)
    onnx_ok = True
    print("ONNX export OK | opset:", model_proto.opset_import[0].version,
          "| ops:", sorted({n.op_type for n in model_proto.graph.node}))
except ImportError as e:
    print("onnx/onnxscript not installed (", e.name, ") -> `pip install onnx onnxscript onnxruntime`; skipping export")
except Exception as e:
    print("ONNX export failed:", type(e).__name__, str(e)[:200])

onnx/onnxscript not installed ( onnx ) -> `pip install onnx onnxscript onnxruntime`; skipping export


### 10. Compare ONNX Runtime output with eager (guarded)

**What this cell does (plain language):** if the ONNX file was produced and `onnxruntime` is installed, we run it on three batch sizes (verifying the dynamic axis works) and compare with the eager outputs. Differences around 1e-7 to 1e-6 are normal float rounding because different kernels sum in different orders.

In [10]:
if not onnx_ok:
    print("skipped: no ONNX file was produced above")
else:
    try:
        import onnxruntime as ort, numpy as np
        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        for bs in (1, 5, 32):
            xi = torch.randn(bs, 8)
            out_ort = sess.run(None, {"x": xi.numpy()})[0]
            with torch.no_grad(): out_eager = net(xi).numpy()
            print(f"batch {bs:>2}: max abs diff eager vs onnxruntime = {np.abs(out_ort - out_eager).max():.2e}")
    except ImportError:
        print("onnxruntime not installed -> `pip install onnxruntime`; skipping runtime comparison")

skipped: no ONNX file was produced above


### 11. Does export make it faster? A (noisy) timing comparison

**What this cell does (plain language):** we time eager, traced and scripted versions of the MLP on a moderately sized batch. For small models the differences are tiny and noisy: export is primarily about portability, and speed-ups come from fusion/optimized runtimes on bigger models. Always measure on your own hardware.

In [11]:
def bench(f, xi, n=200):
    with torch.no_grad():
        for _ in range(20): f(xi)            # warm up (TorchScript profiles during the first runs)
        t = time.perf_counter()
        for _ in range(n): f(xi)
    return (time.perf_counter() - t) / n * 1e6
xi = torch.randn(64, 8)
for name, f in (("eager", net), ("traced", tr_net), ("scripted", torch.jit.script(net)), ("exported", ep.module())):
    print(f"{name:9s} {bench(f, xi):8.1f} us/call")

eager         21.4 us/call
traced        15.6 us/call
scripted      13.6 us/call
exported      93.2 us/call


## Common bugs

- **Tracing a model with data-dependent `if`/`for` and trusting the result.** Fix: Use `torch.jit.script` / `torch.cond`, or restructure with tensor ops (`torch.where`); and always compare against eager on inputs that hit each branch.
- **Exporting in `train()` mode.** Fix: Call `model.eval()` first; otherwise dropout/batch-norm statistics are baked into the graph.
- **Hard-coded batch size or sequence length after export.** Fix: Declare dynamic axes (`dynamic_axes` for ONNX, `Dim` for `torch.export`) and test several shapes.
- **Verifying with only the example input used at export time.** Fix: Use `torch.testing.assert_close` on varied inputs and batch sizes, with a sensible tolerance (1e-5 for fp32).
- **Unsupported ops or Python features (`print`, dict mutation, third-party calls) in scripted code.** Fix: Refactor into tensor-only code, or `@torch.jit.ignore`/`@torch.jit.unused`; consider tracing the submodule and scripting the wrapper.
- **Forgetting that `torch.jit.load` needs the same torch version family.** Fix: Pin versions between export and deployment, and test the loaded artifact in the target environment.
- **Assuming ONNX supports every PyTorch op.** Fix: Check the exporter error, try a newer opset, or replace the op with a supported equivalent.
- **Ignoring numerical differences between runtimes.** Fix: Expect ~1e-6 deviations in fp32; compare with tolerances, not `==`.
- **Treating TorchScript as the long-term path for new projects.** Fix: TorchScript is in maintenance mode; prefer `torch.export` (and ONNX via the dynamo exporter) for new work.

## Exercises

**Exercise 1.** Trace `Net` with an example input of batch size 2, then call the traced module on batch size 11. Does it work? Why (not a pitfall for plain Linear layers)?

In [12]:
# Your attempt for Exercise 1 here

**Solution 1**

Linear layers do not hard-code the batch size, so it generalizes; the pitfall appears only when Python code reads shapes and builds constants.

In [13]:
# Solution 1
t = torch.jit.trace(net, torch.randn(2, 8))
print(t(torch.randn(11, 8)).shape)

torch.Size([11, 4])


**Exercise 2.** Script a function (not a module) `def clamp_relu(x, hi: float): ...` that returns `x.clamp(0, hi)` if `x.mean() > 0` else zeros, and test both branches.

In [14]:
# Your attempt for Exercise 2 here

**Solution 2**

In [15]:
# Solution 2
@torch.jit.script
def clamp_relu(x: torch.Tensor, hi: float):
    if x.mean() > 0:
        return x.clamp(0.0, hi)
    return torch.zeros_like(x)
print(clamp_relu(torch.tensor([1.0, 5.0]), 2.0), clamp_relu(torch.tensor([-1.0, -5.0]), 2.0))

tensor([1., 2.]) tensor([0., 0.])


**Exercise 3.** Use `torch.jit.trace` on `net` and print its graph (`traced.graph`). Count how many `aten::linear` nodes appear.

In [16]:
# Your attempt for Exercise 3 here

**Solution 3**

Two Linear layers should show up as two `aten::linear` nodes.

In [17]:
# Solution 3
g = str(tr_net.inlined_graph)   # .graph keeps submodule calls opaque; inlined_graph expands them
print(g.count("aten::linear"))

2


**Exercise 4.** Save the traced model to a file in a temp dir, reload with `torch.jit.load(map_location='cpu')`, and verify equality with eager.

In [18]:
# Your attempt for Exercise 4 here

**Solution 4**

In [19]:
# Solution 4
p = os.path.join(tempfile.mkdtemp(), "t.pt")
tr_net.save(p)
r = torch.jit.load(p, map_location="cpu")
xi = torch.randn(3, 8)
print(torch.allclose(r(xi), net(xi)))

True


**Exercise 5.** Export `net` with `torch.export` using a dynamic batch `Dim` and show that a call with batch 100 (above the declared max of 64 in section 8) is rejected.

In [20]:
# Your attempt for Exercise 5 here

**Solution 5**

Dynamic dims carry the declared min/max; calls outside the range violate the exported guard.

In [21]:
# Solution 5
try:
    ep.module()(torch.randn(100, 8))
except Exception as e:
    print("rejected:", type(e).__name__)

rejected: AssertionError
